### rc_cleaned_8

Contains the final, working implementation of the Echo State Network which was then applied to various locations. 

- Queensland Government, & Australian Bureau of Metreorology. (1996). SILO - Australian climate data from 1889 to yesterday [Data set]. In Using spatial interpolation to construct a comprehensive archive of Australian climate data. https://doi.org/10.1016/S1364-8152(01)00008-1    

-- To note: the zone-based metrics were computed for the Grains Research and Development Corporation defined 'sub-regions' of the WA Agro-Ecological Zones.

--- Date:   August 2026

In [15]:
import urllib.request
import urllib.parse
import pandas as pd
import io
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd


In [16]:
def collect_data(form,lat,lon,start,finish,username="email@gmail.com",password='apirequest'):
    """
    quering and requesting data from SILO using API
    """
    params = {'format': form,
              'lat': lat,
              'lon': lon,
              'start': start,
              'finish': finish,
              'username': username,
              'password': password
             }

    api_url = 'https://www.longpaddock.qld.gov.au/cgi-bin/silo' 
    url = api_url + '/DataDrillDataset.php?' + urllib.parse.urlencode(params)
    
    with urllib.request.urlopen(url) as remote:
        data = remote.read()
        body_str = data.decode('utf-8')
        
    return pd.read_csv(io.StringIO(body_str), skiprows=31, sep=r'\s+')



In [17]:
def clean_data(df):
    """
    converting requested dataset into usable pandas dataset
    - extracting dates in format dd/mm/yyyy
    - converting columns to numeric
    - computing cumulative growing degree days using tmin, tmax and tbase (10)
    """
    df = df.drop(index=0).reset_index(drop=True)
    df['Date2'] = pd.to_datetime(df['Date2'], format='%d/%m/%Y')

    # convert columns to numeric 
    for cols in df.columns:
        if cols != 'Date2':
            try:
                df[cols] = pd.to_numeric(df[cols])
            except (ValueError, TypeError):
                pass

    # extract year, month, day
    df['Year'] = df['Date2'].dt.year
    df['Month'] = df['Date2'].dt.month
    df['Day'] = df['Date2'].dt.day
    # apply GDD calculation
    df['T.Avg'] = (df['T.Max'] + df['T.Min']) / 2
    df['GDD'] = np.where(df['T.Avg'] >= 10, df['T.Avg'] - 10, 0)
    # find cumulative growing degree days
    df['GDD.C'] = df['GDD'].cumsum()
    
    return df

def prepare_data(df, by):
    """
    preparing rainfall and cumulative growing degree set variables 
    - perform smoothing on rainfall observations through rolling window average
    -
    - normalise rainfall observations
    - extract normalised rain and gddc vectors
    """

    clean = clean_data(df)
    # smooth rain values using rolling window average
    clean['Rain.Mean'] = clean.Rain.rolling(window=180,min_periods=1).mean()

    # if wanting to retain normal time based spacing (extracts a weekly point)
    if by == "Time":
        clean_rain = clean[['Rain.Mean']].values[::7]
        clean_gddc = clean[['GDD.C']].values[::7]
    else:
        # generate regular gdd poitns every 100 units
        clean_gddc = np.arange(0, clean.GDD.sum(), 100)
        # estimate rainfall at each of these gdd points
        clean_rain = np.interp(clean_gddc, clean["GDD.C"], clean["Rain.Mean"])

    normalised_rain = []
    min_val = min(clean_rain)
    max_val = max(clean_rain)
    diff = max_val - min_val # normalise through min-max scaling
    for x in clean_rain:
        normalised_rain.append((x - min_val) / diff)
    return np.array(normalised_rain).reshape(len(clean_gddc),1), clean_gddc.reshape(len(clean_gddc),1)




In [18]:
class ESN:
    """
    N: number of nodes 
    d: average number of connections per node
    rho: spectral radius
    split: proportion of data to take as training  
    """
    def __init__(self, N, d, rho, input_scaling):
        self.N = N
        self.d = d
        self.rho = rho
        self.a = input_scaling


        # random, sparse, reservoir weights:
        ## connections between nodes decided by probability of d/N
        # sampling from normal distribution provides random weight values.
        Weights = np.multiply(
            np.random.binomial(size=(self.N,self.N), p=self.d/self.N, n=1),
            np.random.normal(size=(self.N,self.N)))
        
        # rescale so that rho = largest absolutel eigenvalue (maintaining echo state property)
        self.Vadj = Weights * (self.rho/np.max(np.absolute(np.linalg.eig(Weights)[0])))

        # input matrix: maps the scalar input to every node of the reservoir
        self.Vin = self.a * np.random.uniform(-1,1,size=(1, self.N))

        # bias matrix: bias added to each node's activation
        self.Vbias = np.random.uniform(-1,1,size=(1,self.N))

    def run_reservoir(self,input_data):
        """
        drive the reservoir using the input_data and return the state at every timestep
        states = (len(input_data), N)
        - comments keep track of matrix multiplications
        """
        states = np.zeros((len(input_data)+1, self.N)) # (timesteps x N)
        for i in range(1, len(input_data)+1): # start at 1 to access previous reservoir state
            states[i, :] = np.tanh(
                np.dot(states[i-1:i, :], self.Vadj) + # (1,N) x (N,N) = (1,N)
                np.dot(input_data[i-1:i], self.Vin) +  # (1,1) x (1,N) = (1,N)
                self.Vbias)
        return states[1:,:] # (timesteps x N), skip first row as it stays zero due to no input being processed. 
      

    def train(self, train_input, train_target, washout, beta=0.01):
        """
        fit the output weights
        - remove the initial washout period from the states and target 
        - compute Vout through ridge regression
        """
        train_states = self.run_reservoir(train_input)
        # removing the washout period 
        train_states = train_states[washout:,:] 
        train_target = train_target[washout:,]

        # using ridge regression to compute the output weights through training
        self.Vout = np.dot(
            np.linalg.pinv(np.dot(train_states.T,train_states) + beta*np.identity(self.N)), #(N,t) x (t,N) = (N,N) + (N,N)
                           np.dot(train_states.T, train_target)) #(N,t) x (t,1) = (N,1)
                           #Vout = (N,1)

        return self.Vout
                           

    def freerun(self, test_input, steps):
        """
        method to perform freerun prediction
        - train the reservoir computer on some testing input data 
        - following training, the readout prediction replaces the input data
        - ESN will form predictions just using previous predictions, with no new information being processed.  
        """
        readout = self.run_reservoir(test_input) 
        freerun_states = np.zeros((steps, self.N))
        freerun_states[0,:] = readout[-1,:]
        for i in range(1, steps):
            freerun_states[i, :] = np.tanh(np.dot(freerun_states[i-1:i, :],self.Vadj) + #(1,N) x (N,N) = (1,N)
                                           np.dot(np.dot(freerun_states[i-1:i], self.Vout), self.Vin) + 
                                           self.Vbias) # the prediction replaces input_data[i-1:i]
                                           
        return np.dot(freerun_states, self.Vout)
                     

    def predict(self, test_input, test_target,washout=10):
        """
        run reservoir on test data and map states to predictions
        - remove initial washout period from states and testing input
        - compute Root Mean Squared Error of predictions 
        """
        test_states = self.run_reservoir(test_input)

        # apply trained linear readout
        predictions = np.dot(test_states, self.Vout)

        return predictions, test_states
    
    def metrics(self, test_input, test_target, washout=10):
        """
        method to return computed Root Mean Squared Error (RMSE) and readout vector to plot.
        - generate predictions
        - compute RMSE 
        """
        predictions, test_states = self.predict(test_input, test_target)
        rmse = np.sqrt(((predictions[washout:,] - test_target[washout:,])**2).mean()) # compute RMSE
        complexity = np.log(np.trapezoid((np.abs(np.ravel(self.Vout))))) # compute complexity


        return predictions, rmse, complexity # return predictions, rmse and complexity



In [ ]:
def splitting_data(rain,gddc,step):
    """
    function to split rainfall and cumulative growing degree days by step size
    80% of data for training, 20% for testing
    """
    # step to decide how far prediction goes
    train_size = int(len(rain) * 0.8)
    train_input = rain[:train_size-step]
    train_target = rain[step:train_size]
    test_input = rain[train_size:-step]
    test_input_x = gddc[train_size:-step]
    test_target = rain[train_size+step:]
    test_target_x = gddc[train_size+step:]

    return train_input, train_target, test_input, test_input_x, test_target, test_target_x

def plotting_data(x_values, test_actual, test_predictions, location, step, time):
    """
    function to plot the resulting predictions against the actual data
    - now adjusted to take in both gdd spacing and time spacing
    """
    plt.plot(x_values,test_actual, label = 'actual_values',color='blue',)
    plt.plot(x_values, test_predictions, label='predicted',color='green',alpha=0.6)
    if time == True:
        plt.xlabel('Time in Days')
        title = 'Reservoir Computer Prediction of Rainfall against Days for \n' + str(location) + ' at ' + str(step) + ' Steps'
        plt.title(title)
    else:
        plt.xlabel('Cumulative Growing Degree Days (°C Days)')
        title = 'Reservoir Computer Prediction of Rainfall against Cummulative GDD. for \n' + str(location) + ' at ' + str(step) + ' Steps'
        plt.title(title)
    plt.ylabel('Standardised Rain (mm)')
    plt.title(title)
    plt.legend(['Actual Values', 'Prediction'])
    plt.savefig(f'{location}_time={time}_step={step}')
    plt.show()


def calculate_metrics(esn, location, rain, gddc, step, washout=100):
    """
    function to generate ESN-based predictions and compute RMSE, 
    also returns readout vector to see node weightages assigned during training. 
    """
    train_input, train_target, test_input, test_input_x, test_target, test_target_x = splitting_data(rain, gddc, step)
    esn.train(train_input, train_target, washout=washout)
    predictions, rmse, complexity = esn.metrics(test_input, test_target)

    return rmse, complexity

def rc_for_location(esn, location, rain, gddc, step, plotting, time=False, washout=100):
    """
    function run the ESN prediction and return RMSE  with a plot if requested.
    """
    train_input, train_target, test_input, test_input_x, test_target, test_target_x = splitting_data(rain, gddc, step)
    esn.train(train_input, train_target, washout=washout)
    rain_pred, rmse = esn.predict(test_input, test_target)
    if plotting == True:
        plotting_data(test_target_x[10:], test_target[10:], rain_pred[10:], location=location, step=step, time=time)

    return rmse


def rc_with_freerun(esn, location, rain, gddc, step, freerun_steps, time=False):
   train_input, train_target, test_input, test_input_x, test_target, test_target_x = splitting_data(rain, gddc, step)
   esn.train(train_input, train_target, washout=100)
   free_run_test = esn.freerun(train_target, freerun_steps)

   plt.plot(test_input[:freerun_steps], label = 'actual_values',color='blue',)
   plt.plot(free_run_test, label='predicted',color='green',alpha=0.6)
   if time == True:
       plt.xlabel('Time in days')
       plt.title(f'Freerun Prediction of Rainfall against days at Step {freerun_steps} for Merredin')
   else:
       plt.xlabel('Cumulative Growing Degree Days (°C Days)')
       plt.title(f'Freerun Prediction of Rainfall against Cumulative Growing Degree Days\n at Step {freerun_steps} for Merredin')

   plt.ylabel('Standardised Rain (mm)')
   plt.legend(['Actual Values', 'Freerun Prediction'])
   plt.show()

   return free_run_test


In [ ]:
# dictionary with 10 locations for each of the GRDC defined sub-regions, with ids, latitude and longitude.

zones = {
    "Kwinana East Port Zone": {
        "Merredin":     {"id": 10092, "lat": -31.48, "lon": 118.28},
        "Noongaar":     {"id": 12064, "lat": -31.33, "lon": 118.97},
        "Wialki":       {"id": 10135, "lat": -30.49, "lon": 118.13},
        "Wongan Hills": {"id": 8137,  "lat": -30.89, "lon": 116.72},
        "Bullfinch":    {"id": 12011, "lat": -30.98, "lon": 119.07},
        "Hines Hill":   {"id": 10151, "lat": -31.53, "lon": 118.10},
        "Codg Codgen":  {"id": 10149, "lat": -31.34, "lon": 117.82},
        "Turkey Hill North":  {"id": 12079, "lat": -31.04, "lon": 119.31},
        "Jourerdine":   {"id": 10192, "lat": -30.55, "lon": 118.28},
        "Wattoning":    {"id": 10030, "lat": -30.78, "lon": 118.20},
    },
    "Kwinana West Port Zone": {
        "Northam":             {"id": 10111, "lat": -31.65, "lon": 116.66},
        "Dalwallinu (North)":  {"id": 8014,  "lat": -30.22, "lon": 116.78},
        "Pingelly":            {"id": 10626, "lat": -32.53, "lon": 117.08},
        "Narrogin":            {"id": 10614, "lat": -32.93, "lon": 117.18},
        "Corrigin":            {"id": 10536, "lat": -32.33, "lon": 117.87},
        "Hakea":               {"id": 8238, "lat": -30.10, "lon": 116.23},
        "Balgerbine":          {"id": 8022, "lat": -30.29, "lon": 116.33},
        "Wannamal":            {"id": 9040, "lat": -31.14, "lon": 116.05},
        "Chelsea":             {"id": 9006, "lat": -30.63, "lon": 115.78},
        "Wagin":               {"id": 10647, "lat": -33.31, "lon": 117.34},

    },
    "Geraldton Port Zone": {
        "Northampton":   {"id": 8100, "lat": -28.36, "lon": 114.63},
        "Mullewa":       {"id": 8095, "lat": -28.54, "lon": 115.51},
        "Three Springs": {"id": 8121, "lat": -29.53, "lon": 115.76},
        "Coorow":        {"id": 8037, "lat": -29.88, "lon": 116.02},
        "Eradu":         {"id": 8200, "lat": -28.69, "lon": 115.04},
        "Coorow":        {"id": 8037, "lat": -29.88, "lon": 116.02},
        "Arena":         {"id": 8273, "lat": -29.36, "lon": 115.45},
        "Green Grove":   {"id": 8057, "lat": -29.55, "lon": 115.07},
        "Lynton":        {"id": 8075, "lat": -28.21, "lon": 114.31},
        "Latham":        {"id": 8072, "lat": -29.76, "lon": 116.44},

    },
    "Esperance Port Zone": {
        "Esperance":           {"id": 9789,  "lat": -33.83, "lon": 121.89},
        "Ravensthorpe":        {"id": 10633, "lat": -33.58, "lon": 120.05},
        "Salmon Gums Res.Stn": {"id": 12071, "lat": -32.99, "lon": 121.62},
        "Katta Barega":        {"id": 12312, "lat": -33.59, "lon": 122.45},
        "Erinair":             {"id": 9772,  "lat": -33.76, "lon": 121.22},
        "Peppermint Grove":    {"id": 9594,  "lat": -34.44, "lon": 119.36},
        "Monte Vista":         {"id": 10895,  "lat": -33.40, "lon": 119.93},
        "Scaddan West":        {"id": 12198,  "lat": -33.38, "lon": 121.45},
        "Lort River":          {"id": 12223,  "lat": -33.22, "lon": 121.35},
        "Chesalon":            {"id": 10878,  "lat": -33.56, "lon": 119.58},
        "Richfields":          {"id": 12114,  "lat": -33.68, "lon": 120.70},
    },
    "Albany Port Zone": {
        "Albany":     {"id": 9500,  "lat": -35.03, "lon": 117.88},
        "Illalangi":  {"id": 9523,  "lat": -34.97, "lon": 117.27},
        "Cranbrook":  {"id": 10537, "lat": -34.30, "lon": 117.55},
        "Kojonup":    {"id": 10582, "lat": -33.83, "lon": 117.16},
        "Cape Riche": {"id": 9520,  "lat": -34.61, "lon": 118.75},
        "Manjimup":   {"id": 9573,  "lat": -34.25, "lon": 116.14},
        "Walpole Forestry": {"id": 9805,  "lat": -34.98, "lon": 116.73},
        "Redmond":    {"id": 9678,  "lat": -34.88, "lon": 117.67},
        "Kukerin":    {"id": 10665,  "lat": -33.19, "lon": 118.08},
        "Newdegate":  {"id": 10617,  "lat": -33.09, "lon": 119.02},             
    },
}



In [ ]:
np.random.seed(123)

esn = ESN(300,10,1.0,1.0)

# appending the rain and cumulative gdd variables to the zone dictionary. 
for zone, locations in zones.items():
    for station, info in locations.items():
        data = collect_data(form='standard', lat=str(info["lat"]), lon = str(info["lon"]), start='19710101', finish='20260327')
        info["rain"], info["cgdd"] = prepare_data(data, by=None)

# computing the metrics of each of the locations. 
metrics = []
for zone, locations in zones.items():
    for station, info in locations.items():
        for step in (1, 10, 100):
            rmse, complexity = calculate_metrics(esn = esn, rain= info["rain"], gddc= info["cgdd"], step=step)
            metrics.append({
                'Zone': zone,
                'Location': station,
                'Step': step,
                'RMSE' : rmse,
                'Complexity': complexity
            })
   
metrics_df = pd.DataFrame(metrics)

print(metrics_df.head())
metrics_df.shape
            

KeyboardInterrupt: 